# Food Competition Hierarchy Analysis

## Goal
Determine whether male and female CD1 cages differ in dominance hierarchy structure using the Directional Consistency Index (DCI).

Hierarchy metrics are calculated from pairwise winner-loser interactions pooled across all food competition trials.

In [98]:
import HierarchiaPy
import os
import h5py
import numpy as np
import pandas as pd
import scipy as sp
import matplotlib.pyplot as plt
from scipy.signal import butter, filtfilt, resample_poly
from scipy.signal import butter, filtfilt, resample_poly, resample, find_peaks
import neurokit2 as nk
from scipy.signal import find_peaks
import seaborn as sns
from scipy.stats import wilcoxon
import scipy

# Pandas display settings (optional)
pd.set_option("display.max_rows", None)
pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", None)
pd.set_option("display.width", 0)
print(HierarchiaPy.__version__)

0.2.6


In [99]:
file = r"C:\Users\sjs93\Downloads\CD1_food_comp_all_data_model 1 (1).xlsx"
xls = pd.ExcelFile(file)

print(xls.sheet_names)

['Food_competition_D1_FEMALE', 'Food_Competition_D2_FEMALE', 'Food_Comp_D1_MALE', 'Food_Comp_D2_MALE']


In [100]:
df = pd.read_excel(
    file,
    sheet_name="Food_competition_D1_FEMALE"
)

print(df.head())
print(df.columns)

     Day 2 Cage 1 Unnamed: 2 Unnamed: 3  Unnamed: 4 Unnamed: 5 Cage 2  \
0      NaN  match    Winners     Losers         NaN        NaN  match   
1  trial 1   2vs3        1.3        1.2         NaN    trial 1   2vs3   
2  trial 1   4vs1        1.1        1.4         NaN    trial 1   4vs1   
3  trial 1   3vs4        1.3        1.4         NaN    trial 1   3vs4   
4  trial 1   2vs1        1.1        1.2         NaN    trial 1   2vs1   

  Unnamed: 7 Unnamed: 8  Unnamed: 9 Unnamed: 10 Cage 3 Unnamed: 12  \
0    Winners     Losers         NaN         NaN  match     Winners   
1        2.3        2.2         NaN     trial 1   2vs3         3.3   
2        2.4        2.1         NaN     trial 1   4vs1         3.4   
3        2.4        2.3         NaN     trial 1   3vs4         3.3   
4        2.2        2.1         NaN     trial 1   2vs1         3.1   

  Unnamed: 13  Unnamed: 14 Unnamed: 15 Cage 4 Unnamed: 17 Unnamed: 18  \
0      Losers          NaN         NaN  match     Winners      Lose

In [76]:
print(dir(HierarchiaPy))

['Hierarchia', 'HierarchiaPy', '__builtins__', '__cached__', '__doc__', '__file__', '__loader__', '__name__', '__package__', '__path__', '__spec__', '__version__', 'methods', 'metrics', 'name', 'utilities']


In [109]:
print(hier.dci())
print(hier.landau_h())
print(hier.davids_score())

0.4444
{'Improved_Landau_h': 0.2, 'p_value_r': 0.6238, 'p_value_l': 0.3762}
{1.1: 2.0, 1.2: -0.6667, 1.3: -0.6667, 1.4: -0.6667}


In [110]:
female_cols = {
    1: (2,3),
    2: (7,8),
    3: (12,13),
    4: (17,18),
    5: (22,23),
    6: (27,28)
}

In [111]:
male_cols = {
    2: (2,3),
    3: (7,8),
    5: (12,13),
    7: (17,18),
    8: (22,23)
}

# ============================================================
# Helper Function
# ============================================================
# Extract winner-loser interactions from a specific cage.
# Each row represents one food competition trial.
#
# Inputs:
#   df          = sheet dataframe
#   winner_col  = winner column index
#   loser_col   = loser column index
#   start_row   = first row containing trial data
#
# Output:
#   dataframe with columns:
#       winner
#       loser
# ============================================================

In [93]:
# ============================================================
# Extract winner-loser data from a cage
# ============================================================

def get_cage_data(df, winner_col, loser_col, start_row):
    cage = df.iloc[start_row:, [winner_col, loser_col]].copy()

    cage.columns = ["winner", "loser"]

    cage = cage.dropna()

    return cage

In [ ]:
# def get_cage_data(df, winner_col, loser_col, start_row=1):

#     cage = df.iloc[start_row:, [winner_col, loser_col]].copy()
#     cage.columns = ["winner", "loser"]

#     cage = cage.dropna()

#     return cage

In [86]:
# def get_cage_data(df, winner_col, loser_col, start_row=1):

#     cage = df.iloc[start_row:, [winner_col, loser_col]].copy()
#     cage.columns = ["winner", "loser"]

#     cage = cage.dropna()

#     return cage

In [105]:
d1_f = pd.read_excel(file, sheet_name="Food_competition_D1_FEMALE")
d2_f = pd.read_excel(file, sheet_name="Food_Competition_D2_FEMALE")

results = []

for cage_num, (wcol, lcol) in female_cols.items():

    cage_d1 = get_cage_data(d1_f, wcol, lcol, start_row=1)

    # D2 has an extra header row
    cage_d2 = get_cage_data(d2_f, wcol, lcol, start_row=2)

    combined = pd.concat([cage_d1, cage_d2], ignore_index=True)

    hier = Hierarchia(combined, "winner", "loser")

    results.append({
        "cage": cage_num,
        "sex": "Female",
        "DCI": hier.dci(),
        "Landau_h": hier.landau_h()["Improved_Landau_h"]
    })

female_results = pd.DataFrame(results)

print(female_results)


   cage     sex     DCI  Landau_h
0     1  Female  0.3889       0.1
1     2  Female  0.7778       1.0
2     3  Female  0.5556       1.0
3     4  Female  0.3889       0.9
4     5  Female  0.6111       0.9
5     6  Female  0.4444       0.9


In [89]:
# Female Cage 3

cage_d1 = get_cage_data(d1_f, 12, 13, start_row=1)
cage_d2 = get_cage_data(d2_f, 12, 13, start_row=2)

combined = pd.concat([cage_d1, cage_d2], ignore_index=True)

hier = Hierarchia(combined, "winner", "loser")

print(hier.landau_h())

{'Improved_Landau_h': 1.0, 'p_value_r': 0.0, 'p_value_l': 1.0}


In [90]:
print("DCI:", hier.dci())
print("Landau:", hier.landau_h())
print("David's:", hier.davids_score())

DCI: 0.5556
Landau: {'Improved_Landau_h': 1.0, 'p_value_r': 0.0, 'p_value_l': 1.0}
David's: {3.3: 4.0, 3.1: 1.3333, 3.4: -1.3333, 3.2: -4.0}


In [106]:
print(combined)

   winner loser
0     6.2   6.3
1     6.1   6.4
2     6.3   6.4
3     6.1   6.2
4     6.1   6.3
5     6.4   6.2
6     6.1   6.3
7     6.2   6.3
8     6.2   6.4
9     6.1   6.4
10    6.1   6.2
11    6.4   6.3
12    6.2   6.1
13    6.3   6.2
14    6.3   6.4
15    6.4   6.2
16    6.1   6.3
17    6.1   6.4
18    6.3   6.2
19    6.1   6.4
20    6.2   6.4
21    6.1   6.3
22    6.3   6.4
23    6.1   6.2
24    6.4   6.3
25    6.1   6.2
26    6.2   6.3
27    6.4   6.1
28    6.2   6.4
29    6.1   6.3
30    6.1   6.4
31    6.2   6.3
32    6.2   6.4
33    6.1   6.3
34    6.4   6.3
35    6.2   6.1


In [116]:
# ============================================================
# Load Male Sheets
# ============================================================

d1_m = pd.read_excel(
    file,
    sheet_name="Food_Comp_D1_MALE"
)

d2_m = pd.read_excel(
    file,
    sheet_name="Food_Comp_D2_MALE"
)

print(d1_m.shape)
print(d2_m.shape)

(19, 24)
(19, 24)


In [118]:
for cage_num in male_cols.keys():

    cage_d1 = get_cage_data(
        d1_m,
        *male_cols[cage_num],
        start_row=1
    )

    cage_d2 = get_cage_data(
        d2_m,
        *male_cols[cage_num],
        start_row=1
    )

    combined = pd.concat([cage_d1, cage_d2])

    bad = combined[
        combined["winner"] == combined["loser"]
    ]

    if len(bad) > 0:
        print(f"\nCage {cage_num}")
        print(bad)

In [114]:
print(male_cols)

{2: (2, 3), 3: (7, 8), 5: (12, 13), 7: (17, 18), 8: (22, 23)}


In [112]:
for cage_num in female_cols.keys():

    cage_d1 = get_cage_data(d1_f, *female_cols[cage_num], start_row=1)
    cage_d2 = get_cage_data(d2_f, *female_cols[cage_num], start_row=2)

    combined = pd.concat([cage_d1, cage_d2])

    bad = combined[combined["winner"] == combined["loser"]]

    if len(bad) > 0:
        print(f"\nCage {cage_num}")
        print(bad)